**Basic User Registration Form**

Use Case: Validating user input from a web form.

In [1]:
from pydantic import BaseModel, EmailStr, Field
from typing import Optional

In [ ]:
#!pip install pydantic[email]

In [ ]:
class UserRegistration(BaseModel):
    # Required fields with validation
    username: str = Field(min_length=3, max_length=20, pattern=r"^[a-zA-Z0-9_]+$")
    email: EmailStr  # Ensures it's a valid email format
    password: str = Field(min_length=8)

    # Optional field with a default value
    newsletter_opt_in: bool = False

# Notes:

#What is Field()?
#In Pydantic, Field() is a helper function that you use inside a BaseModel to:
#Add validation rules (like min_length, max_length, pattern).
#Provide metadata (like a description, example, title).
#Set default values (alternative to just =).

In [5]:
# Example usage
user_data = {
    "username": "alice123",
    "email": "alice@example.com",
    "password": "securepass123"
}

In [6]:
user = UserRegistration(**user_data)
print(f"Welcome, {user.username}!")
# Output: Welcome, alice123!

Welcome, alice123!


# **Product Catalog Item**

Use Case: Modeling data for an e-commerce system.

In [8]:
from pydantic import BaseModel, Field, HttpUrl
from typing import List
from datetime import datetime

In [ ]:
class Product(BaseModel):
    id: int
    name: str = Field(min_length=1, max_length=100)
    price: float = Field(gt=0, description="Price must be positive")
    in_stock: bool = True
    tags: List[str] = []  # List of categories/tags
    image_url: HttpUrl  # Must be a valid URL

# Notes--------
# tags: List[str] = []
# A list of strings.
# Default: empty list ([]).
# Used for product categories or labels.
# Allowed -> ["electronics", "laptop", "gaming"]
# Not Allowed -> [123, True] (wrong types inside list).

In [16]:
# Example usage
product_data = {
    "id": 101,
    "name": "Coffee Mug",
    "price": 12.99,
    "tags": ["kitchen", "drinks"],
    "image_url": "https://example.com/mug.jpg"
}


In [17]:

product = Product(**product_data)
print(f"{product.name}: ${product.price}")
# Output: Coffee Mug: $12.99

Coffee Mug: $12.99


# **API Response Wrapper**

Use Case: Standardizing the format of API responses.

In [18]:
from pydantic import BaseModel
from typing import Optional, Any

In [ ]:
class APIResponse(BaseModel):
    success: bool
    data: Optional[Any] = None  # Can be any type of data
    message: Optional[str] = None
    error_code: Optional[int] = None

# Notes :
# 1 # data: Optional[Any] = None
# Optional field (default: None).
# Type: Any → can hold any type of data (dict, list, string, number, another model, etc.).
# Example uses:
# If the request succeeded → contains returned data.
# If it failed → usually left as None.

In [27]:
# Example usage for success
success_response = APIResponse(
    success=True,
    data={"user_id": 123, "name": "Alice"},
    message="User created successfully"
)


In [28]:
# Example usage for error
error_response = APIResponse(
    success=False,
    message="User not found",
    error_code=404
)


In [42]:
print(success_response.model_dump())

{'success': True, 'data': {'user_id': 123, 'name': 'Alice'}, 'message': 'User created successfully', 'error_code': None}


In [29]:
print(error_response.model_dump())

{'success': False, 'data': None, 'message': 'User not found', 'error_code': 404}


# **Structured Response from OpenAI**

Use Case: Getting consistent, validated JSON responses from OpenAI instead of free-form text.

In [30]:
from pydantic import BaseModel, Field
from typing import List
from openai import OpenAI


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = (
    "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
)
client = OpenAI()

In [ ]:
class Recipe(BaseModel):
    title: str
    ingredients: List[str] = Field(..., min_length=1)
    steps: List[str] = Field(..., min_length=1)
    preparation_time_minutes: int = Field(ge=1)  # At least 1 minute

# Notes :
# ingredients: List[str] = Field(..., min_length=1)
# Must be a list of strings.
# Field(..., min_length=1):
# ... → means it’s required (cannot be omitted).
# min_length=1 → the list must contain at least one item.
# Example:
# ✅ ["flour", "sugar", "eggs"]
# ❌ [] (empty list → invalid)

In [33]:
# Ask OpenAI to generate a recipe in JSON format
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{
        "role": "user",
        "content": "Give me a simple chocolate chip cookie recipe in JSON format with title, ingredients, steps, and preparation_time_minutes"
    }],
    response_format={"type": "json_object"}
)



In [34]:
recipe_data = response.choices[0].message.content

In [35]:
recipe_data

'{\n  "title": "Classic Chocolate Chip Cookies",\n  "ingredients": [\n    "1 cup unsalted butter, softened",\n    "1 cup granulated sugar",\n    "1 cup packed brown sugar",\n    "2 large eggs",\n    "1 teaspoon vanilla extract",\n    "3 cups all-purpose flour",\n    "1 teaspoon baking soda",\n    "1/2 teaspoon salt",\n    "2 cups semisweet chocolate chips"\n  ],\n  "steps": [\n    "Preheat oven to 350°F (175°C) and line a baking sheet with parchment paper.",\n    "In a large mixing bowl, cream together the butter, granulated sugar, and brown sugar until smooth.",\n    "Beat in the eggs one at a time, then stir in the vanilla extract.",\n    "In a separate bowl, combine the flour, baking soda, and salt. Gradually add this dry mixture to the wet ingredients until well combined.",\n    "Stir in the chocolate chips.",\n    "Drop spoonfuls of dough onto the prepared baking sheet and bake for 10-12 minutes, or until the edges are lightly golden.",\n    "Allow the cookies to cool on the bakin

In [36]:
recipe = Recipe.model_validate_json(recipe_data)

In [37]:
recipe

Recipe(title='Classic Chocolate Chip Cookies', ingredients=['1 cup unsalted butter, softened', '1 cup granulated sugar', '1 cup packed brown sugar', '2 large eggs', '1 teaspoon vanilla extract', '3 cups all-purpose flour', '1 teaspoon baking soda', '1/2 teaspoon salt', '2 cups semisweet chocolate chips'], steps=['Preheat oven to 350°F (175°C) and line a baking sheet with parchment paper.', 'In a large mixing bowl, cream together the butter, granulated sugar, and brown sugar until smooth.', 'Beat in the eggs one at a time, then stir in the vanilla extract.', 'In a separate bowl, combine the flour, baking soda, and salt. Gradually add this dry mixture to the wet ingredients until well combined.', 'Stir in the chocolate chips.', 'Drop spoonfuls of dough onto the prepared baking sheet and bake for 10-12 minutes, or until the edges are lightly golden.', 'Allow the cookies to cool on the baking sheet for a few minutes before transferring them to a wire rack to cool completely.'], preparation

In [38]:
print(f"Recipe: {recipe.title}")

Recipe: Classic Chocolate Chip Cookies


In [39]:
print(f"Prep time: {recipe.preparation_time_minutes} minutes")

Prep time: 30 minutes


In [40]:
print(f"Ingredients: {len(recipe.ingredients)} items")

Ingredients: 9 items


In [41]:
# you can catch error
# Parse and validate the response
try:
    recipe_data = response.choices[0].message.content
    recipe = Recipe.model_validate_json(recipe_data)
    print(f"Recipe: {recipe.title}")
    print(f"Prep time: {recipe.preparation_time_minutes} minutes")
    print(f"Ingredients: {len(recipe.ingredients)} items")
except Exception as e:
    print("Failed to parse recipe:", e)

Recipe: Classic Chocolate Chip Cookies
Prep time: 30 minutes
Ingredients: 9 items
